# sft_math module



Questa libreria usa come base __numpy__, pertanto il suo uso dovrà essere compatibile con numpy.

**typing.py**

Vengono definiti degli TypeAlias utili per imporre uno standard comunque di precisione macchina nel calcolo:
- **Int**: quello che nella libreria sarà definito essere un indice
- **Float**: quello che nella libreria sarà definito essere numero reale
- **Index**: array di indici
- **Array**: array di numeri reali

**class_templates.py**

Vengono definite classi astratte che saranno la base delle classi definite nei moduli.
- **ClassBase**: Questa classe contiene:
    - __constants__(attribute): una lista di tipo **list[Tuple[Callable, str, str]]** contenente:
        1. una funzione che eventualmente trasforma l'attributo prima di mostrarlo (come mostrare gradi anziché i radianti)
        2. il nome dell'attributo da mostrare
        3. una stringa post fissa (che magari indica l'unità di misura)
    - **__repr__** e **__str__** mostrano il contenuto interpretando **__constants__

**geoloc.py**

- **GeoLoc(ClassBase)**: Permette la geolocalizzazione su una sfera fornendo le coordinate esatte della geolocalizzazione
    - **to_gps**(method): trasforma una coordinata pixel in coordinata gps
    - **from_gps**(method): trasforma una coordinata gps in coordinata px

In [2]:
from sft_math import GeoLoc
from sft_math.cam import Camera
from sft_math.cam.px2body_matrix import load_px2body_matrix_from_yaml
PX2BODY_MATRIX = load_px2body_matrix_from_yaml("sft_config.yaml")
from sft_math.cam.affine import Affine
from sft_math.geo import GeoGPS
from sft_math.geo.polar import GPS

cam = Camera(
    16,
    2.74,
    (5328, 3040),
    (2664, 1520),
    (   # la coordinata x in px sarà la coordinata y in coordinate corpo,
        # la coordinata y sarà la coordinata -x in coordinate corpo
        PX2BODY_MATRIX
    )
)

conf = cam.Configuration(
    position=Affine((0,0,-35), ((),(),())),
    roll=15,
    pitch=18,
    yaw=20,
)

gps = GeoGPS(GPS(45, 11))

geoloc = GeoLoc(cam, conf, gps)
print(repr(geoloc))

gps_target = GeoGPS(GPS(45.0001, 11.0001))
print(repr(gps_target))

px_target = geoloc.from_gps(gps_target.gps)
print(repr(px_target))

# compute the gps of the target
gps_target_recomputed = geoloc.to_gps(px_target)
print(repr(gps_target_recomputed))

sft_math.geoloc.GeoLoc(
	camera=Camera(Projection([0.17125]mm/px), (array([5328.]), array([3040.]))px, Affine(0, 2, [[2664. 1520.]], (1,))px, (1,)),
	configuration=Configuration(Affine(0, 3, [[  0.   0. -35.]], (1,)), [15.]°, [18.]°, [20.]°),
	gps=GeoGPS(GPS([45.]°, [11.]°, (1,)), [6371008.8]m, (1,)),
	shape=(1,)
)
sft_math.geo.geo.GeoGPS(
	gps=GPS([45.0001]°, [11.0001]°, (1,)),
	radius=[6371008.8]m,
	shape=(1,)
)
sft_math.cam.affine.affine.Affine(
	rank=0,
	dimension=2,
	vector=[[4845.37808936 1240.96000318]],
	shape=(1,)
)
sft_math.geo.geo.GeoGPS(
	gps=GPS([45.0001]°, [11.0001]°, (1,)),
	radius=[6371008.8]m,
	shape=(1,)
)


**maps.py**

- **Maps(ClassBase)**: Permette di analizzare una mappa, quindi l'astrazione di un'immagine con specifiche relative alla sua posizione sul pianeta
    - **map**(method): trasforma una coordinata GPS in coordinata pixel
    - **unmap**(method): trasforma una coordinata pixel in coordinata gps

In [3]:
from sft_math import Maps
from sft_math.geo import GeoGPS
from sft_math.geo.polar import GPS

manifold_center = GeoGPS(GPS(45, 11))
mapped_points = (
    GPS([46, 44, 44], [10, 10, 12]),
    [[100, 100], [100, 200], [200, 200]],
)

# create a map
myMap = Maps(
    manifold_center,
    mapped_points,
)

pxcoos = myMap.map(GPS(46, 12))
print(pxcoos)
print(myMap.unmap(pxcoos))


[[199.99991139 100.        ]]
GPS([45.99111926]°, [12.0177587]°, (1,))


**maploc.py**

- **MapLoc(ClassBase)**: Permette di geolocalizzare un punto su una mappa usando calcoli in $\mathbb{R}^3$
    - **map**(method): questo metodo produce un affine in coordinate ned (traducibili in un contesto di mappe)
    - **unmap**(method): questo metodo produce un affine in coordinate px (prese da una mappa)

In [8]:
# in questo test si confrontano i metodi (geomap e geoloc)
# definiamo un (fake) geomap che consiste in geoloc + map
from sft_math import MapLoc
from sft_math.geo import GeoGPS

# definisco una fotocamera
gps_scatto = GPS(41.7377843055556,12.5147220277778)
gps_manifold = GeoGPS(GPS( 41.737955,  12.514082))
ne: GeoGPS.Vector = gps_manifold.map(gps_scatto)  # type: ignore
cam = Camera(
    16,
    2.74,
    (5320, 3032),
    (2660, 1516),
    (   # la coordinata x in px sarà la coordinata y in coordinate corpo,
        # la coordinata y sarà la coordinata -x in coordinate corpo
        PX2BODY_MATRIX
    )
)
conf = cam.Configuration(
    position=Affine((ne.nord.item(),ne.east.item(),-25), ((),(),())),
    roll=-8.1,
    pitch=2.2,
    yaw=-62.2,
)
geoloc = GeoLoc(cam, conf, gps)

# considero un punto su px
px_target = (3095, 992)
gps_atteso = GeoGPS(GPS(41.7377991, 12.514721))

# localizzo con geoloc
px_point = Affine(px_target, ((),()))
gps_target = geoloc.to_gps(px_point)
print(gps_target)
print(gps_target - gps_atteso.gps)
# mappo:
ne_target_fromgeoloc = gps.map(gps_target.gps)
print(ne_target_fromgeoloc)

# localizzo con maploc
maploc = MapLoc(cam, conf)
ne_target_aff = maploc.map(px_point)
ne_target_frommaploc = GeoGPS.Vector(ne_target_aff.vector[0,0], ne_target_aff.vector[0,1])
print(ne_target_frommaploc)

gps_geo = gps.unmap(ne_target_fromgeoloc)
gps_map = gps.unmap(ne_target_frommaploc)

print(GeoGPS(gps_map) - gps_geo)

GeoGPS(GPS([41.73784175]°, [12.51471817]°, (1,)), [6371008.8]m, (1,))
Movement([-2.82975904]°, [4.74866978]m, (1,))
Vector([-12.59270807]m, [52.78557632]m, (1,))
Vector([-12.59232902]m, [52.78571265]m, (1,))
Movement([19.78172657]°, [0.09493543]m, (1,))


## geo module


**class_templates.py**

Vengono definite le classi base usate dal modulo geo:
- **GPSBase(ClassBase)**: Una classe GPSBase dovrà contenere:
    - **Movement**(classe): rappresenta un movimento lungo una direzione della sfera, al suo interno troviamo i metodi:
        - **\_\_neg\_\_**: restituisce il movimento opposto 
        - **to_vector**: restituisce il movimento in un manifold
    - **Vector**(classe): rappresenta un movimento in un manifold, in esso troviamo definiti i metodi:
        - **to_movement**: restituisce il movimento nella sfera
    - **\_\_sub\_\_**: calcola il movimento da compiere per cambiare coordinata gps
    - **\_\_add\_\_**: calcola la coordinata gps dopo un movimento
    - **map**(method): data una coordinata gps calcola la coordinata nel manifold
    - **unmap**(method): calcola la coordinata gps dalle coordinate nel manifold

**constants.py**

Vengono definite le costanti nel modulo:
- **EARTH_RADIUS**: raggio terrestre

**geo.py**

- **GeoGPS(GPSBase['GPS', 'GPS'])**: Questa classe implementa le coordinate gps in un pianeta

In [ ]:
from sft_math.geo.polar import *
from sft_math.geo import *

rome = GPS(41.8992, 12.5450)
milan = GPS(45.4613, 9.1595)

# otteniamo il movimento da Roma a Milano
movement = GeoGPS(milan) - rome
print(repr(movement))

# otteniamo il movimento nel manifold
vector = movement.to_vector()
print(repr(vector))

#! altre funzionalità sono analoghe a quelle della classe GPS

### polar module

**polar.py**

- **GPS(GPSBase['GPS', 'GPS'])**: Questa classe implementa le coordinate gps come sistema di riferimento indipendente dal raggio.

In [ ]:
from sft_math.geo.polar import *

# creazione di una coordinata gps:
gps = GPS(38.313912, -57.848098)

# stampa delle coordinate gps utilizzando repr e str definite in ClassBase:
print(repr(gps))
print(str(gps))

In [ ]:
# creazione di un movimento:
gps_dest = GPS(38.303912, -57.858098)  # nuova posizione: ridotta la latitudine e la longitudine

# calcolo il movimento nelle coordinate polari
movement = gps_dest - gps

# yaw: è l'angolo tra il nord e la direzione del movimento
# alpha: è l'angolo del movimento rispetto il centro della Terra
print(repr(movement))

# trasformo il movimento in un vettore nel manifold
v = movement.to_vector()

# dlat: è la variazione della latitudine nel manifold
# dlon: è la variazione della longitudine nel manifold
print(repr(v))

# trasformo il vettore in un movimento, quindi ne confermo la coincidenza
print(repr(v.to_movement()))
# sommo gps a movement per avere gps_dest e confermare la correttezza del movimento
print(repr(gps + movement))

In [ ]:
# Uso del manifold

# creazione di un vettore nel manifold
v1 = GPS(0.001, 0.002).asvector()  # la variazione della latitudine locale è 0.001 mentre quella della longitudine è 0.002

# calcolo la coordinata gps di destinazione
gps_dest = gps.unmap(v1)
print(repr(gps_dest))

# calcolo la coordinata gps di destinazione usando il movimento
gps_dest = gps + v1.to_movement()
print(repr(gps_dest))

# ricalcolo il vettore nel manifold usando map
print(repr(gps.map(gps_dest)))

In [ ]:
import numpy as np

# altre operazioni di supporto:
print(repr(gps.vec()))  # restituisce il vettore in coordinate ECEF nella sfera di raggio 1
print(repr(gps.ned2ecef()))  # restituisce la matrice di rotazione da NED a ECEF con shape (...,3,3)

# quindi per calcolare il vettore nord in coordinate ECEF:
nord = np.einsum('...ik,...k->...i', gps.ned2ecef(), [1, 0, 0])
print(repr(nord))

**utils.py**

Introduce delle funzioni utili per lavorare su coordinate GPS
- **from_vec**: utile per invertire il metodo vec(), estrae le coordinate GPS da una coordinata ECEF
- **barycenter**: utile per calcolare il baricentro di coordinate gps

## cam module

**cam.py**
-   **Cam(ClassBase)**: definisce la classe per una fotocamera, che contiene:
    -   **Projection(ClassBase)**: definisce l'oggetto geometrico che permette la proiezione, che contiene:
        -   **in_plane**: proietta un Affine in $\mathbb{R}^n$ nell'iperpiano di dimensione $\mathbb{R}^{n-1}$
        -   **out_plane**: produce l'Affine in $\mathbb{R}^n$ che proiettato produce l'affine nell'iperpiano di dimensione $\mathbb{R}^{n-1}$
    -   **Configuration(ClassBase)**:  definisce l'oggetto configurazione che permette di convertire delle coordinate ned in corpo:
        -   **xyz2ned**: converte coordinate xyz in coordinate ned
        -   **ned2xyz**: converte coordinate ned in cordinate xyz

In [ ]:
from sft_math.cam import *
from sft_math.cam.px2body_matrix import load_px2body_matrix_from_yaml
PX2BODY_MATRIX = load_px2body_matrix_from_yaml("sft_config.yaml")
from sft_math.cam.affine import *

cam = Camera(
    16,
    2.74,
    (5328, 3040),
    (2664, 1520),
    (   # la coordinata x in px sarà la coordinata y in coordinate corpo,
        # la coordinata y sarà la coordinata -x in coordinate corpo
        PX2BODY_MATRIX
    )
)

# mostro la camera
print(repr(cam))

# creo una configurazione della fotocamera
conf = cam.Configuration(
    position=Affine((0,0,-35), ((),(),())),
    roll=15,
    pitch=18,
    yaw=20,
)

# mostro la configurazione scelta
print(repr(conf))

# proietto un punto nella foto a terra
point = Affine(
    (500, 1200),
    ((),())
)

# crea l'affine generato dal punto
NEDpoint = cam.px2ned(point, conf)

print(repr(NEDpoint))
print(NEDpoint.orthCoos2globalCoos_matrix)

### affine module

**affine.py**

- **Affine(ClassBase)**: Questa classe definisce un sottospazio affine dello spazio euclideo.

In [ ]:
from sft_math.cam.affine import *

# Definiamo uno sottospazio affine
affine = Affine(
    (1,0,-1,1,1),
    (
        ( 1, 2,-1,),
        ( 2, 0,-2,),
        ( 1,-1, 5,),
        (-1, 1, 2,),
        ( 1, 2, 3,)
    )
)

# sono riportati i dettagli dell'affine: rank, dimensione e un punto di appartenenza
print(repr(affine))

# calcoliamo un punto all'interno dello spazio affine
coordinates = (1,2,-1)
point = affine(coordinates)
print(repr(point))

# proviamo ad usare le operazioni algebriche
affine2 = Affine(
    (1,0,1,-1,-1,),
    (
        ( 3, 3, 1),
        ( 2, 3, 2),
        ( 0, 0, 0),
        (-1, 0, 3),
        (-2,-4, 1),
    )
)

# somma
print(repr(affine + affine2))
# moltiplicazione
print(repr(affine * 2))

# relazioni
addAffine = affine + affine2
print(repr(affine <= addAffine))
print(repr(affine < addAffine))
print(repr(affine == addAffine))
print(repr(affine > addAffine))
print(repr(affine >= addAffine))
print(repr(affine != addAffine))

# operazioni insiemistiche
print(repr(affine & affine2))
print(repr(~affine))

**utils.py**
-   **broadcast_to**: Questa funzione cambia lo shape dell'oggetto affine

## clustering module

**PClustering.py**

Viene definita la classe:
- **PClustering**: questa classe permette di applicare il PClustering:
    - **Data**(classe): questa è la classe che formatta e controlla l'input dei dati da fittare con PClustering. Possiede i seguenti campi:
        - **points**: coordinate di $N$ punti $K$-dimensionali
        - **covs**: matrici di covarianza relative ad ogni dato
        - **covs_inv**: matrici di covarianza invertite
        - **mship**: le appartenenze booleane
        - **mship_idx**: le appartenenze indicate come indici
        - **n_clusters**: numero di cluster
        - **n_samples**: numero di dati
        - **n_dimensions**: numero di dimensioni
    - **\_\_init\_\_**(method): La classe viene inizializzata specificando:
        - la matrice di confusione (n_cluster x n_classi), che indica in $M_{r,c}$ la probabilità che un certo cluster $r$ abbia una classe $c$
        - il p value, che permette di determinare la dimensione finale dei cluster, valori più bassi permettono una maggiore velocità ma ridotta iniziativa del modello, rischiando di incappare in soluzioni indesiderate
        - la fuzziness, che concede al modello la possibilità di assegnare membership praticamente nulle ai dati sempre più volentieri man mano che accresce la fuzziness
        - la soglia (threshold), obbliga il modello ad avere un comportamento booleano nei dintorni dei centroidi, valori troppo alti rischiano di produrre centroidi imprecisi, in quanto il loro circondario può includere dati sporchi o persino finti cluster. Valori troppo bassi potrebbero rendere il modello di puro fuzzy non adatto, producendo partecipazioni fuzzy dei dati incoerenti con la realtà.
    - **fit**: Questa funzione trova i centroidi, le membership fuzzy e una maschera che distingue dati inlier e outlier. Il suo input saranno i dati, il numero massimo di iterazioni e la confidenza di uno step, che rappresenta in un certo senso il learning rate. Se si sceglie una confidenza bassa saranno compiute molte modifiche in uno step, questo migliora l'iniziativa, ma rischia di portare troppo velocemente a convergenza il metodo.

In [ ]:
from sft_math.clustering import PClustering

# carico i dati scritti in assets
import numpy as np
conf_M = np.load('assets/conf_M.npy')
points = np.load('assets/points.npy')
covs = np.load('assets/covs.npy')
classes = np.load('assets/classes.npy')

# creo il modello
model = PClustering(
    conf_M,
    p_value=0.002,
    fuzziness=8.0,
    threshold=0.6
)
print(repr(model))

# creo i dati
data = PClustering.Data(
    points,
    covs,
    classes,
    n_clusters=4,
)

print(repr(data))

# aderisco il modello ai dati
ctrs, mship, mask = model.fit(
    data,
    max_iter = 10000,
    step_confidence = 0.98
)

# mostro i centroidi trovati
print(ctrs)

# i centroidi veri sono:
# [ 74.4377537    1.21821388]
# [-37.94035402  -6.65324996]
# [ 21.23349074  -0.65534142]
# [-17.80577048   7.34761673]]

# mostro il numero di dati inlier stimati
print(f"{np.sum(mask)}/{len(mask)}")

## Maps e GeoLoc

Bisogna vedere se Maps e GeoLoc sono coerenti.
Test1:
- inserisco 4 punti della mappa a formare un quadrato 1mx1m
- trasformo i 4 punti in punti GPS
- trasformo i 4 punti GPS in punti px

In [ ]:
from sft_math.geo.polar import GPS
from sft_math.geo import GeoGPS
from sft_math.cam.affine import Affine
from sft_math.cam import Camera
from sft_math.cam.px2body_matrix import load_px2body_matrix_from_yaml
PX2BODY_MATRIX = load_px2body_matrix_from_yaml("sft_config.yaml")
from sft_math import GeoLoc

manifold_center = GeoGPS(GPS(45, 11))

# costruisco un passo verso nord
step_north = GeoGPS.Vector(
    nord=1,
    east=0,
)
step_east = GeoGPS.Vector(
    nord=0,
    east=1,
)

gps_north = manifold_center + step_north.to_movement()
gps_east = manifold_center + step_east.to_movement()

print(repr(manifold_center))
print(repr(gps_north))
print(repr(gps_east))

# creo la fotocamera
cam = Camera(
    16,
    2.74,
    (5320, 3032),
    (2660, 1516),
    (   # la coordinata x in px sarà la coordinata y in coordinate corpo,
        # la coordinata y sarà la coordinata -x in coordinate corpo
        PX2BODY_MATRIX
    )
)
geoloc = GeoLoc(
    cam,
    cam.Configuration(
        position=Affine((0, 0, -35), ((), (), ())),
        roll=0,
        pitch=0,
        yaw=0,
    ),
    manifold_center,
)

print(repr(cam))
print(repr(geoloc))

# proietto i punti gps nella foto
px_north = geoloc.from_gps(gps_north)
px_east = geoloc.from_gps(gps_east)
px_center = geoloc.from_gps(manifold_center.gps)
print("in foto")
print(f"{px_center=}")
print(f"{px_north=}, {px_east=}")

in foto
px_center=sft_math.cam.affine.affine.Affine(
	rank=0,
	dimension=2,
	vector=[[6031.38843322 1516.        ]],
	shape=(1,)
)
px_north=sft_math.cam.affine.affine.Affine(
	rank=0,
	dimension=2,
	vector=[[6031.3884332  1323.34923264]],
	shape=(1,)
), px_east=sft_math.cam.affine.affine.Affine(
	rank=0,
	dimension=2,
	vector=[[6257.57346339 1515.99999998]],
	shape=(1,)
)


Coerenza con CV2:
Supponiamo ora di prendere una mappa generica e una fotocamera

proiettiamo i 4 vertici della foto a terra ottenendo i GPS e quindi i punti della mappa

produciamo così una matrice di trasformazione in CV2 da porzione di mappa a rettangolo

Prendiamo un punto della foto.

Riproiettiamolo a terra con GeoLoc e otteniamo i GPS

Riproiettiamolo in mappa con l'omografia e quindi otteniamo i GPS


In [ ]:
from sft_math.maps import Maps

# genero i 4 vertici a terra

geoloc = GeoLoc(
    cam,
    cam.Configuration(
        position=Affine((0, 0, -90), ((), (), ())),
        roll=30,
        pitch=0,
        yaw=0,
    ),
    manifold_center,
)

vertices_incam = Affine(
    [
        [0.0, 0.0],
        [cam.image_shape[0].item(), 0.0],
        [cam.image_shape[0].item(), cam.image_shape[1].item()],
        [0.0, cam.image_shape[1].item()],
    ],
    ((), ())
)

vertices_gps = geoloc.to_gps(vertices_incam)
print("in gps")
print(f"{vertices_gps=}")

# calcolo sulla mappa i gps


in gps
vertices_gps=sft_math.geo.geo.GeoGPS(
	gps=GPS([45.00032921 45.00019211 44.99980789 44.99967076]°, [10.99839579 10.99988959 10.99988959 10.99839581]°, (4,)),
	radius=[6371008.8 6371008.8 6371008.8 6371008.8]m,
	shape=(4,)
)

